PROJECT OVERVIEW

In this project, I will use a customer segmentation dataset from Kaggle to build a ML model that predicts the customer segment (Segmentation: A, B, C, or D) for new potential customers of an automobile company. The dataset reflects a real-world scenario where the company aims to apply its successful segmentation-based marketing strategy from existing markets to new regions by classifying customers based on their demographic and behavioral attributes.

Task 1: Understanding the dataset

In [ ]:
# Loading the customer segmentation dataset from a CSV file

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

#loading the dataset
Cust_data = pd.read_csv('Customer_Data.csv')


The dataset contains 8068 rows and 11 columns
Each row represents a unique customer and includes:

- Features (Input Variables):
  - Demographic: `Gender`, `Ever_Married`, `Age`, `Graduated`, `Profession`
  - Behavioral: `Work_Experience`, `Spending_Score`, `Family_Size`, `Var_1`
  - Identifier: ID
-Target Variable: Segmentation`: One of four customer groups — A, B, C or D

Task 1: EDA

In [ ]:
# Task 1: Data Understanding

print("Shape of dataset:", Cust_data.shape)

Cust_data.head()

# Step 1: Data Overview
print("\nColumns:")
print(Cust_data.columns)

Cust_data = Cust_data.drop(columns=['ID'])# Dropping unncessary columns

print("\nColumns:")
print(Cust_data.columns)

# Step 2: Target Variable Distribution amongst A,B,C, and D
print("\nClass distribution in Customer dataset:")
print(Cust_data['Segmentation'].value_counts())

# Step 3: Plot target distribution
sns.countplot(x='Segmentation', data=Cust_data, palette="pastel")
plt.title('Distribution of Segmentation Classes')
plt.savefig("Task_1_figs/Distribution of Segmentation Classes.png")
plt.show()



The ID column was deleted after the examination of the dataset since it was only a unique identifier of no predictive power to be used in a model. Value counts and histogram was then used to investigate the distribution of the target variable Segmentation that included A, B, C and D classes. The output revealed a fairly equal class distributions, which informs that the dataset can be used in multi-class classification without being rebalanced.

In [ ]:
# Step 4: Feature Analysis 

# Separate numerical and categorical columns from dataset
numeric_cols = Cust_data.select_dtypes(include='number').columns.tolist() # e.g., Age, Spending Score,Family_Size, Work_Experience
categorical_cols = Cust_data.select_dtypes(include='object').columns.tolist() # e.g., Gender, Profession, Spending_Score, Var_1, Ever_Married

# Summary statistics for numeric columns
print("\nSummary Statistics for Numeric Columns:")
print(Cust_data[numeric_cols].describe())

# Display of value counts for categorical feature to understand dominant categories
print("\nValue Counts for Categorical Columns:")
for col in categorical_cols:
    print(f"\n{col} value counts:")
    print(Cust_data[col].value_counts())

# Step 5: Feature-Target Relationships 

# Boxplots for numeric features vs target
for col in ['Age', 'Work_Experience', 'Family_Size']:
    plt.figure(figsize=(5, 3))
    sns.boxplot(x='Segmentation', y=col, data=Cust_data)
    plt.title(f'{col} by Segment')
    plt.savefig(f"Task_1_figs/{col} by Segment.png")
    plt.show()

# Crosstabs for categorical features vs target

segment_colors = sns.color_palette("Set2", 4)
segment_order = ['A', 'B', 'C', 'D']  # Consistent segment order for all visualizations
for col in categorical_cols:
    if col == 'Segmentation':
        continue
    cross_tab = pd.crosstab(Cust_data[col], Cust_data['Segmentation'], normalize='index')
    cross_tab.plot(kind='bar', stacked=True, color=segment_colors)
    plt.title(f'{col} vs Segmentation')
    plt.ylabel('Proportion')
    plt.xticks(rotation=45)
    plt.savefig(f"Task_1_figs/{col} vs Segmentation.png") # Save plots to folder
    plt.show()


After recognizing the target variable, the exploration of a dataset was continued by differentiating between numerical and categorical features. The numerical features (Age, Work_Experience, Family_Size) were summarized as a number of mean, standard deviation and quartiles, whereas categorical features (Gender, Profession, Spending_Score, and graduation.) were summarized by high counts of values. It was in this first analysis of the features that we got a clear picture of the value distributions and the types of data as well, which will be used as a base of determining the connection of these data points to the labels of the segmentation.

In order to see how the features could be related to the target variable Segmentation, an array of visual analyses were conducted:
* Boxplots of numerical features by segment revealed important trends.Segment D seems to be composed of younger customers and segment A and C are composed of older ones. Likewise, in Segment D, Work_Experience is lower, i.e. the customer base is less experienced. Family_size is (normally) lower in Segment A and more disparate in other segments. Such differences indicate that demographic characteristics can affect the grouping of the segments.

Crosstab-based stacked bar charts for categorical features showed clear segmentation patterns:
* There is no particular gender bias since gender distribution is relatively evenly distributed in all these segments.

* Ever_Married status highlighted that the unmarried customers are concentrated in Segment D whereas married are more prevalent in Segments B and C.

* Segment A, B and C are dominated by graduated people compared to segment D that has non-graduates making a larger portion of the segment.

* An important differentiating factor is profession: Segment D has more homemakers and employees in the healthcare industry and an overrepresentation of professionals such as engineers and executives is observed in Segments A and B.

* Spending_Score tells that Segment C mainly involves high spenders whereas Segment D is related to low spenders.

* Var_1 is an anonymized categorical variable, which has uneven distribution indicating that segment C prevails over the other categories namely Cat_6 whereas segment D is prevalent in segment Cat_3.

In [ ]:
# Boxplots to check outliers
for col in ['Age', 'Work_Experience', 'Family_Size']:
    plt.figure(figsize=(5, 3))
    sns.boxplot(x=Cust_data[col])
    plt.title(f'Outliers in {col}')
    plt.savefig(f"Task_1_figs/Outliers in {col}.png")
    plt.show()

# Step 6: Data Quality Check

# Check for missing values in each column
print("\nMissing values in training data:")
print(Cust_data.isnull().sum())

Cust_data = Cust_data.dropna()

print("\nCheck for Missing values in training data after cleaning:")
print(Cust_data.isnull().sum())

In order to evaluate the spread of the data and identify outliers, the boxplots of the three most relevant numeric variables, Age, Work_Experience, and Family_Size were created.

Age: There is a decent symmetric distribution with some premium extremes above the overage of 85. Most of the data is clustered in a range of 25 to 60 years which are the norms of the usual customer age.

Work_Experience: A few outliers are observed to have more than 10 years of experience and the median value is low (~2 years), indicating that there is a high chance of a significant number of customers belonging to the category of young career-first persons. This is skewed to the right and it can be transformed.

Family_Size: Majority values are between 1-6 with some very big outliers in 7 and above. This pair of extreme values can be indicative of joint families or anomaly of data input

In step 6, nulls check discovered that there were some nulls, which would be dropped with the row-drop nulls approach (dropna). The dataset was cleaned and the missing values were, therefore, ensured after cleaning and the next approach in modeling the data was observed to be of quality.

Task 2: DATA PREPARATION & MODELLING

In [ ]:
# Task 2: Data Preparation & Modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Copy after cleaning before preprocessing
data_train_cleaned = Cust_data.copy()
print("Before Impute")
print(Cust_data)

# Identify categorical and numerical columns
cat_cols = data_train_cleaned.select_dtypes(include='object').columns
num_cols = data_train_cleaned.select_dtypes(include='number').columns

# Impute missing values: use most frequent value for categorical and median for numerical
imputer_cat = SimpleImputer(strategy='most_frequent')
imputer_num = SimpleImputer(strategy='median')
data_train_cleaned[cat_cols] = imputer_cat.fit_transform(data_train_cleaned[cat_cols])
data_train_cleaned[num_cols] = imputer_num.fit_transform(data_train_cleaned[num_cols])

print("After Impute, Before encoding")
print(data_train_cleaned)

# Encode categorical variables using Label Encoder
le = LabelEncoder()
for col in cat_cols:
    data_train_cleaned[col] = le.fit_transform(data_train_cleaned[col])

# Standardize numeric features
scaler = StandardScaler()
data_train_cleaned[num_cols] = scaler.fit_transform(data_train_cleaned[num_cols])

print("After encoding")
print(data_train_cleaned)

# Check for missing values
print("\nMissing values in preprocessed data:")
print(data_train_cleaned.isnull().sum())

# Split features and target
X = data_train_cleaned.drop('Segmentation', axis=1)
y = data_train_cleaned['Segmentation']
y = LabelEncoder().fit_transform(y)  # Encode target labels

# Split: 60% Train, 20% Val, 20% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Save as CSVs
X_train.to_csv("train_features.csv", index=False)
X_val.to_csv("validation_features.csv", index=False)
X_test.to_csv("test_features.csv", index=False)
pd.DataFrame(y_train).to_csv("train_labels.csv", index=False)
pd.DataFrame(y_val).to_csv("validation_labels.csv", index=False)
pd.DataFrame(y_test).to_csv("test_labels.csv", index=False)

Preparation and Preprocessing of Data
The dataset has been preprocessed to undergo modeling by the following main stages:

Cleaning and imputation:
The most frequent category was used to fill in missing values in the categorical features, and medians were used in the case of numerical feature. This maintains the distribution of the data and decreases the effect of outliers.

Encoding Categorical variables:
Label Encoding was used to convert categorical features into numerical. The reason why this approach was preferred over one-hot encoding is that this approach is efficient and models such as Random Forest and XGBoost can handle such a format.

Feature Scaling:
The standardization of numerical characteristics was implemented with the normalization through z-score transformation and StandardScaler, making the algorithms that are sensitive to the scale of features, like Logistic Regression and SVM, efficient.

Train-Validation-Test Split:
The data was divided into 60%, 20%, and 20% trains, validations, and tests respectively by stratified sampling to ensure that the classes were balanced. To help in reproducibility, the splits were saved as CSV files.

Saving the train, teast, and validations files:
> The encoded and scaled input features like Age, Gender, Profession etc. can be found in feature files (train_features.csv, validation_features.csv, test_features.csv).
> Label files (train_labels.csv, validation_labels.csv, test_labels.csv) contain the target variable Segmentation, encoded into numeric form (e.g., A=0, B=1, etc.).
Separation of features and labels will give clarity and flexibility of training and evaluation of models.

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)  # Train model on training data

print("Logistic Regression Evaluation")
print(f"Accuracy     : {accuracy_score(y_train, log_reg.predict(X_train)):.4f}")

joblib.dump(log_reg, "models/log_reg.joblib") # Save model in folder

# Decision Tree Classifier

dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)

print("Decision Tree Classifier Evaluation")
print(f"Accuracy     : {accuracy_score(y_train, dt.predict(X_train)):.4f}")
joblib.dump(dt, "models/dt.joblib")
# Random Forest Classifier

rf = RandomForestClassifier(
    max_depth=10,
    n_estimators=200,
    class_weight='balanced', # To handle class imbalance
    random_state=42,
    n_jobs=-1  # Use all processors for faster training
)
rf.fit(X_train, y_train)

print("Random Forest Classifier Evaluation")
print(f"Accuracy     : {accuracy_score(y_train, rf.predict(X_train)):.4f}")
joblib.dump(rf, "models/rf.joblib")
# Support Vector Machine Classifier

svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=False,
    random_state=42
)
svm.fit(X_train, y_train)

print("SVM Classifier Evaluation")
print(f"Accuracy: {accuracy_score(y_train, svm.predict(X_train)):.4f}")
joblib.dump(svm, "models/svm.joblib")

# Naive Bayes Classifier

nb = GaussianNB()
nb.fit(X_train, y_train)

print("Naive Bayes Classifier Evaluation")
print(f"Accuracy: {accuracy_score(y_train, nb.predict(X_train)):.4f}")
joblib.dump(nb, "models/nb.joblib")

# Train XGBoost model
xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    objective="multi:softmax", # Multi-class classification
    num_class=4, # 4 customer segments: A, B, C, D
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
xgb_model.fit(X_train, y_train)  # y_train is already numeric (0=A, 1=B, etc.)

print("XGBoost Classifier")
print(f"Accuracy:     {accuracy_score(y_train, xgb_model.predict(X_train)):.4f}")
joblib.dump(xgb_model, "models/xgb_model.joblib")



ML models applied to the training data
Model 1: Logistical Regression
It was employed because of its simplicity, interpretability, the capability of dealing with both categorical and continuous predictors. Given the categorical outcome prediction scenario, which is suitable in case of multi-class tasks such as customer segmentation, logistic regression is an ideal task due to its capacity of displaying response through a varied cascade of input features as justified by Buya et al. (2020).

Model 2: Decision Tree
Decision Tree model (max depth=10) was chosen as it would allow modeling complex non-linear relationship because it does not required that its relationships be linear. According to Wang (2024), due to their split-based decision-making procedure, decision trees are widely generalizable and outlier-resistant.

Model 3: Random Forest
The choice of Random Forest fell on the quality of its combination with noise and mixed-type data and its robustness and high accuracy. Because of random selection of the feature and its feature selection algorithm (see Wang, 2024), it constructs several decision trees on bootstrapped data, which lessens overfitting, and enhances generalization, thus it is quite adequate to perform customer segmentation tasks.

Model 4: Support Vector Machine (SVM)
The benefits associated with the use of SVM with an RBF kernel are that it works well with high dimensional space and forms a complex decision surface. It was applied in the framework of the class balancing setup, which also means that it is applicable when there is a need to perform multiple classes in segmentation whose boundaries are subtle.

Model 5: Naive Bayes
It makes the assumption of feature independence, and works well on categorical or low hearted data. Even though simple in nature, it tends to exhibit surprisingly good performance at initial phases of model comparison.

Model 6: XGBoost 
It was chosen due to its efficiency, scalability, and high ability to distinguish one multi-class classification. This indicates that XGBoost could effectively be used in customer churn prediction in the telecommunications industry as Shrestha and Shakya (2022) have achieved the desired results. It has a gradient boosting structure and regularization properties, and is appropriate in customer segmentation tasks wherein the classes are not linearly divided and the data is noisy.

After training the above 6 models on the processed training data, performance was evaluated by calculating the accuracy. 

> Random Forest showed the highest training accuracy (76.5%), which means high learning ability and the opportunity to generalize the information.

> Decision Tree though did (68.4%), however is possibly more susceptible to over fitting.

> XGBoost performed fairly (57.8%), and using tuning is expected to enhance performance.

> The SVM and the Logistic Regression produced lower accuracies (53.4% and 50.2% respectively), indicating that they do not fit the training data very well as is.

> Naive Bayes was the lowest in accuracy (48.5%), because it assumes all features work independently, which may not be true for this data.



TASK 3: EVALUATION

In [ ]:
# Task 3
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import joblib
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier # type: ignore

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

# Model Evaluation Metrics
models = {
    "Logistic Regression": log_reg,
    "Decision Tree": dt,
    "Random Forest": rf,
    "SVM": svm,
    "Naive Bayes": nb,
    "XGBoost Classifier": xgb_model
}

# Define evaluation function to compute performance on train and validation sets

def evaluate(model, name):
    results = {}
    y_predict_train = model.predict(X_train)
    y_predict_val = model.predict(X_val)

    results['Model'] = name
    results['Train Accuracy'] = accuracy_score(y_train, y_predict_train)
    results['Validation Accuracy'] = accuracy_score(y_val, y_predict_val)
    results['F1 Score (Val)'] = f1_score(y_val, y_predict_val, average='weighted')
    results['Precision (Val)'] = precision_score(y_val, y_predict_val, average='weighted')
    results['Recall (Val)'] = recall_score(y_val, y_predict_val, average='weighted')

    # Print classification report and plot confusion matrix
    print(f"\n{name} Classification Report (Validation Set):")
    print(classification_report(y_val,y_predict_val))

    cm = confusion_matrix(y_val, y_predict_val)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix - {name} (Validation Set)")
    plt.savefig(f"Task_3_fig/Confusion Matrix - {name} (Validation Set).png")
    plt.show()
    return results

# To evaluate all models and compile results
results_list = [evaluate(model, name) for name, model in models.items()]
evaluation_df = pd.DataFrame(results_list).sort_values(by='F1 Score (Val)', ascending=False)
display(evaluation_df)

# Identify and evaluate the best model
best_model_name = evaluation_df.iloc[0]['Model']
best_model = models[best_model_name]

print(f"\nBest Model Based on F1 Score: {best_model_name}")

val_preds = best_model.predict(X_val)
print("\nClassification Report (Validation):")
print(classification_report(y_val, val_preds))

# Confusion Matrix for Best Model
cm = confusion_matrix(y_val, val_preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name} (Validation)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig(f"Task_3_fig/Confusion Matrix - {best_model_name} (Validation).png")
plt.show()

# Error Analysis: Misclassified Examples
misclassified_idx = (val_preds != y_val)
misclassified_df = X_val[misclassified_idx].copy()
misclassified_df['Actual'] = y_val[misclassified_idx]
misclassified_df['Predicted'] = val_preds[misclassified_idx]
print("Sample misclassified examples:")
display(misclassified_df.head())

# Hyperparameter Tuning for XGBoost using GridSearchCV
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# Initialize model
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)

# Grid Search for hyper parameter tunning
xgb_grid = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_param_grid,
    scoring='f1_weighted',
    cv=3,
    n_jobs=-1,
    verbose=1
)

# Fit on training data
xgb_grid.fit(X_train, y_train)

# Best estimator
tuned_xgb = xgb_grid.best_estimator_
print(f"\nBest Parameters (XGBoost): {xgb_grid.best_params_}")

# Evaluate tuned model
val_preds = tuned_xgb.predict(X_val)
test_preds = tuned_xgb.predict(X_test)

print(f"Tuned XGBoost F1 Score (Val): {f1_score(y_val, val_preds, average='weighted'):.4f}")
print(f"Tuned XGBoost Accuracy (Test): {accuracy_score(y_test, test_preds):.4f}")

print("\nTuned XGBoost Classification Report (Test):")
print(classification_report(y_test, test_preds))

# Confusion Matrix (Test data)
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, test_preds), annot=True, fmt='d', cmap='Purples')
plt.title('Confusion Matrix - Tuned XGBoost (Test Set)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig(f"Task_3_fig/Confusion Matrix - Tuned XGBoost (Test Set).png")
plt.show()

# Save final model
joblib.dump(tuned_xgb, "models/final_model.joblib")


Task 3:
Metric Selection:
F1-score (weighted), was chosen as the most important scale, because of the class imbalance, since this metric considers both precision and recall. Precision may not be sufficient to cause deception particularly when there are class inadequacy. The accuracy of a naive baseline (e.g. predicting majority always) would be about 26-30%, so all models were expected to achieve a higher accuracy significantly.

Model Evaluation:

XGBoost performed the best in terms of F1-score (0.5298) and its performance in terms of the classes showed to be quite good especially in Classes 2 and 3.

Random Forest was a good fit on the training data however with an indication of overfitting but still generalised well.

SVM had positive generalization as its results were similar in training and validation.

The overfitting of Decision Tree in the training data and unstable results when making predictions on unseen data were observed.

Logistic Regression and Naive Bayes fared worst, most probably because of their assumptions leading to the simplification of a model and the inability to grasp complex non-linear trends.

Class-wise Trends:

The majority of the models were the best estimating Class 3. A,B,C and D = 0,1,2,3 respectively

Class 1 was always hard to predict with high precision and recall windows being the lowest in this class.

These tree based models were more stable on Class 2.

Error Analysis:
Examples of the validation set that were misclassified were identified and reviewed to get an insight on the shortcomings of the model. The top two errors involved Class 0 & 3 and Class 1 & 3, indicating either common patterns in features or lack of discriminatory cues. Certain row results in the validation set demonstrated that when the values of such features as Spending_Score and the Profession had similar values, this led to confusion of the existing predictions. This qualitative analysis made the weaknesses on a class level apparent and preconditioned the tuning of XGBoost.

Model Improvement:
XGBoost hyperparameters tuning was done via the GridSearchCV, and the following best parameters identified were:

Once it was tuned, the F1-score increased a little and validation/test performance became constant, proving to be the most reliable model.

Final Model WITH IMPROVEMENTS:
The final model to be used was XGBoost because of the balance between performance and generalization. Validation F1 rose slightly to 0.5298 and the test accuracy was 0.5371 which is very close to the validation performance and this means that there was minimal overfitting and good generalization with unseen data.


TASK 4: REFLECTION

The objective of this project was to develop a machine learning algorithm that could predict the customer segments (segment A, B, C or D) of an automobile company that was venturing into new markets. The XGBoost model turned out to be the most successful with a weighted F1-score of about 0.53, which is significantly above a naive baseline (~26-30 percent). Although it is not flawless, the model offers a decent initial solution to segmentation-based marketing, which allows reaching individuals. Nevertheless, the regular under-estimation of destinations of Segment B (Class 1), support the idea that the current features might not detect all the segments.

Feature Influence and Model Behavior
By examining confusion matrices and classification reports, it was established that all models and particularly tree-based such as XGBoost and Random Forest fared well on Classes 2 and 3 (Segments C and D), which could be explained by recognition of a more comprehensible behavioral signature. Segment B (Class 1) was however easily confused with other classes, which meant low feature separability or imbalance. This was magnified by the fact that precision and recall of this class was just low in all these grades. Groups significantly misclassified tended to follow a similar patterning in Spending Score or Profession indicating that these features are not empowered enough to differentiate specific segments.

Ethics and Fairness
However, though race and income were not provided in the dataset, they contained such sensitive demographic attributes as Gender, Ever_Married, and Profession. These may serve as proxies to social biases in general. There is also the fact that the model performed consistently poor on Segment B and thus may represent that type of customer or neglect their resources. Fairness measurements and methodologies such as class balancing or feature audits would have to be implemented before the deployment to guarantee equal results.

Model Interpretability
Communication between stakeholders requires interpretability, where the customer segmentation is an important example. Other smaller models, such as Logistic Regression and Decision Tree, could be explained much more easily, but they performed worse than more complicated models. Less transparent XGBoost provided better accuracy.

To keep this trade-off in balance we have explored the feature importances of XGBoost- Spending Score, Profession and Age were most influential. Although more sophisticated tools such as SHAP were not applied, they provided a general idea of how models work. To deploy clarifying tools would be a beneficial addition to provide transparency and trust.

Future Work and Limitation
The key drawbacks of the model are:

> Limitation of the features: There were few demographic and behavioral features which decreased customer diversity that could be covered by the model.

> Lack of balanced performance Some segments (in particular, Segment B) performed not very well on all models, which indicates that there should be more effective feature separation.

> Moderate accuracy: The model exceeded the baseline, but it might not meet the required level of an accurate model in case of a crucial business decision not being augmented.

Improvements can be made in future by:

> Obtaining more detailed or specialized features (i.e., prior purchases, location etc).

> Using interpretability methods such as SHAP on stakeholder transparency and model debugging.

> Investigation of hybrid models or further tuning in order to stretch the performance without over fitting.


REFRENCES

Buya S, Tongkumchum P, Owusu B E. Modelling of land-use change in Thailand using binary logistic regression and multinomial logistic regression. Arabian Journal of Geosciences, 2020, 13: 1 - 12.

Wang, Zhiyue. (2024). Customer Segmentation Based on Machine Learning Methods. Highlights in Science, Engineering and Technology. 92. 126-132. 10.54097/g70xqb16. 

Sagar Maan Shrestha, Aman Shakya, A Customer Churn Prediction Model using XGBoost for the Telecommunication Industry in Nepal,
Procedia Computer Science, Volume 215, 2022, Pages 652-661, ISSN 1877-0509, https://doi.org/10.1016/j.procs.2022.12.067.
(https://www.sciencedirect.com/science/article/pii/S187705092202138X)




